# ARVI-RX - Remote MedGemma API on Colab

This notebook starts a real MedGemma 4B multimodal API on Google Colab GPU.
Your local Streamlit app can call this API with the `remote_medgemma` mode.

This is an educational prototype only, not a medical device.

## 0. Requirements

Use a GPU runtime: `Runtime -> Change runtime type -> GPU`.

You need:
- Hugging Face access to `google/medgemma-4b-it`.
- An ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken.

In [ ]:
!pip -q install -U transformers accelerate torch pillow huggingface_hub fastapi uvicorn python-multipart pyngrok

In [ ]:
import json
import re
import tempfile
import threading
import time
from getpass import getpass
from pathlib import Path

import torch
import uvicorn
from fastapi import FastAPI, File, Form, UploadFile
from huggingface_hub import login
from PIL import Image
from pyngrok import ngrok
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "google/medgemma-4b-it"
PROMPT_VERSION = "remote_medgemma_colab_v1"
WARNING_TEXT = "Prototype pedagogique. Non destine au diagnostic. Validation par un professionnel qualifie requise."
ALLOWED_CLASSES = {"normal", "suspected_opacity", "pneumonia_suspected", "uncertain"}

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("A Colab GPU runtime is required.")
print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
hf_token = getpass("Hugging Face token: ")
login(token=hf_token)
del hf_token

ngrok_token = getpass("ngrok auth token: ")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("Hugging Face and ngrok configured.")

In [ ]:
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("Loading", MODEL_ID, "with", dtype)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map={"": "cuda:0"},
)
model.eval()

if any(getattr(t, "is_meta", False) for t in list(model.parameters()) + list(model.buffers())):
    raise RuntimeError("Model contains meta tensors; weights are not fully loaded.")

print("MedGemma loaded on GPU.")

In [ ]:
STRICT_PROMPT = f"""
Tu es un assistant radiologue virtuel pedagogique pour le projet ARVI-RX.
Analyse cette radiographie thoracique frontale.
Reponds uniquement avec un JSON valide. N'ajoute aucun texte hors JSON.

Schema obligatoire :
{{
  "class": "normal" | "pneumonia_suspected" | "uncertain",
  "confidence": nombre entre 0 et 1,
  "observations": ["observation courte et prudente"],
  "justification": "justification courte et prudente",
  "limits": "limites de l analyse",
  "warning": "message rappelant que ce n est pas un avis medical"
}}

Contraintes :
- Ne donne pas de diagnostic medical definitif.
- Si l'image est ambigue, de mauvaise qualite ou si tu n'es pas sur, utilise "uncertain".
- Les seules classes autorisees sont normal, pneumonia_suspected, uncertain.
""".strip()


def extract_json_object(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        payload = json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if not match:
            raise
        payload = json.loads(match.group(0))
    if not isinstance(payload, dict):
        raise ValueError("Model response JSON is not an object")
    return payload


def normalize_prediction(payload: dict, raw_text: str = "") -> dict:
    predicted_class = str(payload.get("predicted_class") or payload.get("class") or "uncertain")
    if predicted_class not in ALLOWED_CLASSES:
        predicted_class = "uncertain"
    try:
        confidence = float(payload.get("confidence", 0.0))
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(confidence, 1.0))
    evidence = payload.get("visual_evidence") or payload.get("observations") or []
    if isinstance(evidence, str):
        evidence = [evidence]
    limitations = payload.get("limitations") or payload.get("limits") or []
    if isinstance(limitations, str):
        limitations = [limitations]
    result = {
        "image_quality": str(payload.get("image_quality") or "limited"),
        "predicted_class": predicted_class,
        "confidence": round(confidence, 3),
        "visual_evidence": [str(x) for x in evidence] if isinstance(evidence, list) else [],
        "justification": str(payload.get("justification") or "No justification provided."),
        "limitations": [str(x) for x in limitations] if isinstance(limitations, list) else ["remote output requires review"],
        "warning": WARNING_TEXT,
        "model_name": MODEL_ID,
        "prompt_version": PROMPT_VERSION,
        "raw_model_response": raw_text,
    }
    result["class"] = payload.get("class") or ("pneumonia_suspected" if predicted_class == "suspected_opacity" else predicted_class)
    result["observations"] = result["visual_evidence"]
    result["limits"] = "; ".join(result["limitations"])
    return result


def run_medgemma(image_path: str | Path, prompt: str = STRICT_PROMPT) -> dict:
    start = time.perf_counter()
    image = Image.open(image_path).convert("RGB")
    print(f"received image for MedGemma: path={image_path} size={image.size} mode={image.mode} prompt_chars={len(prompt)}")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=prompt_text, images=image, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    raw_text = processor.decode(generated[0][input_len:], skip_special_tokens=True).strip()
    payload = extract_json_object(raw_text)
    result = normalize_prediction(payload, raw_text=raw_text)
    result["latency_ms"] = int((time.perf_counter() - start) * 1000)
    return result

In [ ]:
app = FastAPI(title="ARVI-RX remote MedGemma API")


@app.get("/")
def health():
    return {"status": "ok", "model": MODEL_ID, "scope": "educational prototype, not diagnosis"}


@app.get("/health")
def health_check():
    return health()


@app.post("/predict")
async def predict(file: UploadFile = File(...), prompt: str = Form(STRICT_PROMPT)):
    suffix = Path(file.filename or "image.png").suffix or ".png"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await file.read())
        tmp_path = Path(tmp.name)
    try:
        print(f"/predict received file={file.filename} content_type={file.content_type} bytes={tmp_path.stat().st_size}")
        return run_medgemma(tmp_path, prompt=prompt or STRICT_PROMPT)
    except Exception as exc:
        return {
            "image_quality": "limited",
            "predicted_class": "uncertain",
            "confidence": 0.0,
            "visual_evidence": [],
            "justification": f"Remote MedGemma inference failed: {exc}",
            "limitations": ["remote inference failure", "educational prototype"],
            "warning": WARNING_TEXT,
            "model_name": MODEL_ID,
            "prompt_version": PROMPT_VERSION,
            "error_detail": str(exc),
        }
    finally:
        tmp_path.unlink(missing_ok=True)


PORT = 500
NGROK_DOMAIN = "fragile-suing-goldmine.ngrok-free.dev"
public_url = ngrok.connect(PORT, "http", domain=NGROK_DOMAIN).public_url
print("Remote MedGemma API URL:", public_url)
print("Paste this URL into Streamlit mode remote_medgemma.")

thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info"),
    daemon=True,
)
thread.start()